# ⚽ Futbol Mundial — Tablero de predicciones del Mundial 2026

Modelo que combina **Transfermarkt** (valor de plantel, lesionados, stats por jugador) + **Elo histórico dinámico** (49.000 partidos desde 1872, ya incorpora la racha reciente), con la llave oficial de FIFA. Parámetros calibrados y auditados contra partidos reales (60,7% de acierto out-of-sample, sin sobreconfianza).

## Cómo correrlo de cero

Ejecutá las celdas **en orden** con ▶ (o `Ctrl+Enter`). Cada una dice cuánto tarda.

1. **Setup** — descarga todo y levanta la API (~2-3 min)
2. **Auto-test** — verifica que la API funciona (~10 seg) ← *si falla acá, frená y avisá*
3. **Planteles** — baja los 48 planteles reales (~2 min rápido / ~15-20 min completo)
4. **Simulación** del Mundial 50.000 veces
5-9. **Análisis**: grupos, clasificación, equipo, goleador, partido suelto

In [ ]:
# 1. Setup desde cero: descargar el proyecto, instalar y levantar la API local (~2-3 min)
# Clona de cero para garantizar el código más reciente y un caché limpio.
import os, subprocess, time, shutil
import requests as rq

if os.path.exists('/content/Futbol-mundial'):
    shutil.rmtree('/content/Futbol-mundial')
!git clone -q -b claude/laughing-ritchie-f4b202 https://github.com/GustaPardo/Futbol-mundial.git /content/Futbol-mundial
%cd /content/Futbol-mundial/predicciones
!pip install -q requests
!pip install -q -r ../transfermarkt-api/requirements.txt

# Levantar la API parcheada dentro del propio Colab
!pkill -f 'app/main.py' 2>/dev/null
time.sleep(2)
env = dict(os.environ, PYTHONPATH='/content/Futbol-mundial/transfermarkt-api')
subprocess.Popen(['python', '/content/Futbol-mundial/transfermarkt-api/app/main.py'],
                 env=env, stdout=open('/tmp/api.log', 'w'), stderr=subprocess.STDOUT)

def api_viva():
    try:
        return rq.get('http://localhost:8000/docs', timeout=3).status_code == 200
    except Exception:
        return False

for _ in range(40):
    if api_viva():
        break
    time.sleep(2)

os.environ['TM_API_URL'] = 'http://localhost:8000'
os.environ['TM_API_PAUSA'] = '0.4'   # pausa de cortesía entre requests
print('API local:', 'OK ✅' if api_viva() else '✘ FALLÓ — corré:  !tail -30 /tmp/api.log')

In [ ]:
# 2. Diagnóstico integral (~30 seg) — corré esto SIEMPRE antes de la celda 3
# Prueba los endpoints, lee el log del servidor automáticamente y testea los
# XPath del parser contra las páginas reales de Transfermarkt.
# Si algo no da OK, pegá TODA esta salida en el chat: con eso se arregla de una.
!python diagnostico.py

In [ ]:
# 3. Planteles reales de los 48 mundialistas
#
# MODO "rapido"   -> valor de mercado + edad + lesionados descontados (~2 min)
# MODO "completo" -> además pondera cada jugador por nivel de competencia
#                   (Champions/ligas top), minutos y goles+asistencias de la
#                   última temporada, y habilita el goleador con datos reales
#                   (~15-20 min; tiene caché: si se corta, re-ejecutá y retoma)
MODO = "completo"

flag = "--con-stats" if MODO == "completo" else ""
!python generar_scores.py {flag}

In [ ]:
# 4. Simular el Mundial completo 50.000 veces (~1 min)
# Campeón, finalista y goles esperados por equipo, con la llave oficial FIFA,
# Elo dinámico, localía de anfitriones y forma reciente.
!python simular_mundial.py -n 50000

In [ ]:
# 4b. Predicción de los partidos reales de la fase de grupos (~20 seg)
# Cada partido del fixture en orden cronológico: probabilidades (gana A / empate
# / gana B) y marcador más probable. Cambiá DIAS para ver más o menos jornadas.
DIAS = 0   # 0 = los 72 partidos; ej. 3 = solo los primeros 3 días

!python analizar.py partidos --dias {DIAS}

In [ ]:
# 5. Análisis grupo por grupo (~30 seg)
# Probabilidad de cada equipo de salir 1°, 2° y de clasificar a 16avos.
!python analizar.py grupos -n 20000

In [ ]:
# 6. Cómo llegaron al Mundial (~10 seg) — análisis descriptivo
# Campaña de eliminatorias de cada equipo y su rendimiento vs lo esperado por Elo.
# Nota honesta: la auditoría matemática (auditar_modelo.py) midió que el peso
# óptimo de esta señal en la predicción es 0 — el Elo dinámico ya contiene la
# racha de cada equipo, y sumarla de nuevo empeora el modelo (doble conteo).
!python analizar.py clasificacion

In [ ]:
# 7. Radiografía de un equipo (~30 seg)
# Hasta qué ronda llega, quién lo elimina en cada instancia, su grupo y su forma.
# Nombres en inglés: "Portugal", "Germany", "Iraq", "Argentina", "Mexico"...
EQUIPO = "Portugal"

!python analizar.py equipo "{EQUIPO}" -n 20000

In [ ]:
# 8. Goleador del Mundial, jugador por jugador (~1 min) — necesita la celda 3
# Reparte los goles esperados de cada selección entre sus jugadores reales.
# Con MODO "completo" usa goles/minutos reales de la última temporada.
!python analizar.py goleador -n 20000 --top 20

In [ ]:
# 9. Predicción de un partido suelto (~15 seg)
equipo_a = "Iraq"
equipo_b = "Uzbekistan"
local = ""   # "A" si el primero juega de local, "B" si el segundo, "" neutral

extra = f'--local {local}' if local else ''
!python predictor.py "{equipo_a}" "{equipo_b}" {extra}

### Cómo funciona (y qué se auditó)

- **Fuerza de cada equipo**: 50% Elo histórico (49.000 partidos, dinámico — ya incorpora la racha reciente de cada equipo) + 50% valor de plantel de Transfermarkt (lesionados afuera; en modo completo cada jugador ponderado por nivel de liga, minutos y producción).
- **Modelo de goles Poisson** calibrado por máxima verosimilitud con 11.600 partidos (2010–2022) y validado sobre 3.539 de 2023+ que nunca vio: **60,7% de acierto** en 1X2 (baseline 47%).
- **Auditoría de calibración**: el modelo NO es sobreconfiado — cuando dice 65%, ocurre el 65% de las veces (tabla de confiabilidad por deciles en `docs/ANALISIS.md`). El bonus de "momentum" se probó y se descartó con datos: todo peso > 0 empeora la predicción (doble conteo con el Elo dinámico). Para reproducir la auditoría: `!python auditar_modelo.py`.
- **Simulador**: fixture real de grupos, 8 mejores terceros con restricciones FIFA, llave oficial (partidos 73–104), Elo dinámico, localía de anfitriones, alargue y penales.

Metodología completa y limitaciones: [`docs/ANALISIS.md`](https://github.com/GustaPardo/Futbol-mundial/blob/claude/laughing-ritchie-f4b202/docs/ANALISIS.md).